In [3]:
import os
import sys
os.environ.setdefault("XDG_DATA_HOME", "/ibex/user/sotoorda/masterh1/LLM_project/LLM_model/.xdg_data")
print(sys.executable)
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")))

/ibex/user/sotoorda/conda-environments/crewai_mistral/bin/python
HF_TOKEN set: False


In [4]:
from getpass import getpass
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if not hf_token:
    hf_token = getpass("Enter your Hugging Face token: ")
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
os.environ["HUGGINGFACE_API_KEY"] = hf_token

In [5]:
import os
import json
import pandas as pd
from collections import Counter

# Load the three prediction files
celltypist_path = '/ibex/user/sotoorda/masterh1/LLM_project/CellTypist/predictions_celltypist.csv'
rf_path = '/ibex/user/sotoorda/masterh1/LLM_project/Random_forest/prediction_results.csv'
seurat_path = '/ibex/user/sotoorda/masterh1/LLM_project/Label_transfer/predictions_seurat.csv'

# Output artifacts consumed by the CrewAI report cell
output_dir = '/ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results'
os.makedirs(output_dir, exist_ok=True)
per_cell_csv = os.path.join(output_dir, 'consensus_per_cell_results.csv')
metrics_json = os.path.join(output_dir, 'consensus_metrics.json')


df_ct_raw = pd.read_csv(celltypist_path, index_col='Cell_id')
df_rf_raw = pd.read_csv(rf_path, index_col='Cell_id')
df_sr_raw = pd.read_csv(seurat_path, index_col='Cell_id')

# Keep a minimal aligned view for consensus voting
df_ct = df_ct_raw[['Predicted_label']].copy().rename(columns={'Predicted_label': 'CellTypist'})
df_rf = df_rf_raw[['Predicted_label']].copy().rename(columns={'Predicted_label': 'RandomForest'})
df_sr = df_sr_raw[['Predicted_label']].copy().rename(columns={'Predicted_label': 'Seurat'})

# Ground truth is used later by benchmarking/reporting if available
if 'Ground_truth' in df_ct_raw.columns:
    df_gt = df_ct_raw[['Ground_truth']].copy()
elif 'Ground_truth' in df_rf_raw.columns:
    df_gt = df_rf_raw[['Ground_truth']].copy()
elif 'Ground_truth' in df_sr_raw.columns:
    df_gt = df_sr_raw[['Ground_truth']].copy()
else:
    df_gt = pd.DataFrame(index=df_ct.index, data={'Ground_truth': 'NA'})

common_cells = df_ct.index.intersection(df_rf.index).intersection(df_sr.index).intersection(df_gt.index)
n_common = len(common_cells)

print(f"Common cells found: {n_common}")
if n_common == 0:
    raise ValueError('No common cells found across CellTypist, RandomForest, and Seurat outputs.')

results_df = pd.concat([
    df_gt.loc[common_cells],
    df_ct.loc[common_cells],
    df_rf.loc[common_cells],
    df_sr.loc[common_cells],
], axis=1)
results_df = results_df.reset_index().rename(columns={'index': 'Cell_id'})

consensus_labels = []
vote_supports = []
agreement_classes = []

for _, row in results_df.iterrows():
    votes = [str(row['CellTypist']), str(row['RandomForest']), str(row['Seurat'])]
    count = Counter(votes)
    consensus, support = count.most_common(1)[0]
    consensus_labels.append(consensus)
    vote_supports.append(int(support))
    if support == 3:
        agreement_classes.append('unanimous')
    elif support == 2:
        agreement_classes.append('majority')
    else:
        agreement_classes.append('disagreement')

results_df['Consensus_label'] = consensus_labels
results_df['Vote_support'] = vote_supports
results_df['Agreement_class'] = agreement_classes

results_df.to_csv(per_cell_csv, index=False)

metrics = {
    'n_cells': int(len(results_df)),
    'unanimous_3of3': int((results_df['Vote_support'] == 3).sum()),
    'majority_2of3': int((results_df['Vote_support'] == 2).sum()),
    'disagreement_1of3': int((results_df['Vote_support'] == 1).sum()),
    'pairwise_celltypist_vs_randomforest': float((results_df['CellTypist'] == results_df['RandomForest']).mean() * 100.0),
    'pairwise_celltypist_vs_seurat': float((results_df['CellTypist'] == results_df['Seurat']).mean() * 100.0),
    'pairwise_randomforest_vs_seurat': float((results_df['RandomForest'] == results_df['Seurat']).mean() * 100.0),
}

with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Consensus artifacts created:')
print('-', per_cell_csv)
print('-', metrics_json)
print('\nPreview:')
print(results_df.head(3).to_string(index=False))

Common cells found: 23466
Consensus artifacts created:
- /ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results/consensus_per_cell_results.csv
- /ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results/consensus_metrics.json

Preview:
                  Cell_id Ground_truth CellTypist RandomForest           Seurat Consensus_label  Vote_support Agreement_class
_MO291_AAACAGCCACCACAAC-1         ProB     pre-PC       pre-PC           pre-PC          pre-PC             3       unanimous
_MO291_AAACAGCCATCCATCT-1          HSC        MPP          HSC              HSC             HSC             2        majority
_MO291_AAACATGCACCGTTCC-1          GMP        MPP  Plasma Cell Granulocytic-UNK             MPP             1    disagreement


In [ ]:
import os
import json
import pandas as pd
from crewai import Agent, Task, Crew, LLM, Process

# Paths from the consensus-generation cell
output_dir = '/ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results'
per_cell_csv = os.path.join(output_dir, 'consensus_per_cell_results.csv')
metrics_json = os.path.join(output_dir, 'consensus_metrics.json')

results_df = pd.read_csv(per_cell_csv)
with open(metrics_json, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

# Fixed fact sheet
n_cells = int(len(results_df))
n_unanimous = int((results_df['Vote_support'] == 3).sum())
n_majority = int((results_df['Vote_support'] == 2).sum())
n_disagreement = int((results_df['Vote_support'] == 1).sum())

pct_unanimous = round(100.0 * n_unanimous / n_cells, 2)
pct_majority = round(100.0 * n_majority / n_cells, 2)
pct_disagreement = round(100.0 * n_disagreement / n_cells, 2)

agree_ct_rf = round(100.0 * (results_df['CellTypist'] == results_df['RandomForest']).mean(), 2)
agree_ct_sr = round(100.0 * (results_df['CellTypist'] == results_df['Seurat']).mean(), 2)
agree_rf_sr = round(100.0 * (results_df['RandomForest'] == results_df['Seurat']).mean(), 2)

rf_disagrees_both = int(((results_df['RandomForest'] != results_df['CellTypist']) & (results_df['RandomForest'] != results_df['Seurat'])).sum())
rf_plasma_bias = int(((results_df['RandomForest'] == 'Plasma Cell') & (results_df['CellTypist'] != 'Plasma Cell') & (results_df['Seurat'] != 'Plasma Cell')).sum())

fact_sheet = f"""
FACT_SHEET_DO_NOT_CHANGE_NUMBERS
- n_cells: {n_cells}
- unanimous_3of3: {n_unanimous} ({pct_unanimous}%)
- majority_2of3: {n_majority} ({pct_majority}%)
- disagreement_1of3: {n_disagreement} ({pct_disagreement}%)
- pairwise_celltypist_vs_randomforest: {agree_ct_rf}%
- pairwise_celltypist_vs_seurat: {agree_ct_sr}%
- pairwise_randomforest_vs_seurat: {agree_rf_sr}%
- randomforest_disagrees_with_both_count: {rf_disagrees_both}
- randomforest_plasma_cell_conflict_count: {rf_plasma_bias}
- policy_strict: exclude Vote_support=1; keep Vote_support in [2,3]
- policy_moderate: keep Vote_support=2 and 3, but flag Vote_support=2 as medium-confidence
- policy_permissive: keep all cells; add uncertainty label to Vote_support=1
- confidence_high: Vote_support=3
- confidence_medium: Vote_support=2
- confidence_low: Vote_support=1
- likely_risk_1: method disagreement is concentrated in RandomForest versus the other methods
- likely_risk_2: strong Plasma Cell conflict pattern in RandomForest suggests classifier bias or calibration mismatch
""".strip()

model_name = 'huggingface/mistralai/Mistral-7B-Instruct-v0.2:featherless-ai'

# Run each step in a fresh single-task Crew to avoid cross-call message-state issues

def run_single_crewai_step(role, goal, backstory, prompt, expected_output, temperature=0.0, max_tokens=450):
    llm = LLM(model=model_name, temperature=temperature, max_tokens=max_tokens, stream=False)
    agent = Agent(
        role=role,
        goal=goal,
        backstory=backstory,
        llm=llm,
        verbose=False,
        allow_delegation=False,
    )
    task = Task(description=prompt, expected_output=expected_output, agent=agent)
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=False)
    return crew.kickoff().raw

report_rules = (
    'Use ONLY FACT_SHEET numbers and policy lines. Do not invent any metric. '
    'Do not mention UMI, mitochondrial %, ribosomal %, or doublet score. '
    'If information is unavailable, write: "not available in fact sheet". '
    'Required sections exactly: '
    '1) Summary of reliability '
    '2) Exclusion policy (strict/moderate/permissive with Vote_support thresholds) '
    '3) Confidence tiers for downstream analysis '
    '4) Top risks and likely causes '
    '5) Prioritized next steps.'
)

# Initial draft
initial_prompt = (
    'Create an initial QC report from FACT_SHEET. '
    + report_rules
    + f"\n\n{fact_sheet}"
)

current_report = run_single_crewai_step(
    role='QC Reporter',
    goal='Write factual QC report from fixed evidence.',
    backstory='You are concise and never hallucinate numbers.',
    prompt=initial_prompt,
    expected_output='Initial report with exact FACT_SHEET numbers.',
)

feedback_history = []

# Two review-improve iterations
for iteration in [1, 2]:
    critic_prompt = (
        'Audit REPORT against FACT_SHEET. '
        'Output exactly two sections only:\n'
        'A) FAILED CHECKS\n'
        'B) FIX INSTRUCTIONS\n'
        'Keep it concise and concrete.\n\n'
        f"FACT_SHEET:\n{fact_sheet}\n\nREPORT:\n{current_report[:3500]}"
    )

    feedback = run_single_crewai_step(
        role='QC Critic',
        goal='Detect hallucinations and weak recommendations.',
        backstory='You are a strict audit reviewer and enforce evidence-only writing.',
        prompt=critic_prompt,
        expected_output='Short audit with failed checks and explicit fixes.',
    )
    feedback_history.append(feedback)

    revise_prompt = (
        'Revise REPORT using CRITIC_FEEDBACK and FACT_SHEET. '
        + report_rules
        + f"\n\nFACT_SHEET:\n{fact_sheet}\n\nREPORT:\n{current_report[:3500]}\n\nCRITIC_FEEDBACK:\n{feedback[:2500]}"
    )

    current_report = run_single_crewai_step(
        role='QC Reporter',
        goal='Produce improved evidence-grounded report.',
        backstory='You apply critic feedback exactly and preserve factual accuracy.',
        prompt=revise_prompt,
        expected_output='Improved report satisfying all critic checks.',
    )

    with open(os.path.join(output_dir, f'qc_report_critic_feedback_iter{iteration}.txt'), 'w', encoding='utf-8') as f:
        f.write(feedback)
    with open(os.path.join(output_dir, f'qc_report_revised_iter{iteration}.txt'), 'w', encoding='utf-8') as f:
        f.write(current_report)

# Final synthesis pass after 2 iterations
synthesis_prompt = (
    'Create the final polished report. '
    + report_rules
    + ' Ensure the output is a normal report, not a FAILED CHECKS list. '
      'Each section must include at least 2 concrete bullets and explicit thresholds for Vote_support. '
    + f"\n\nFACT_SHEET:\n{fact_sheet}\n\nLATEST_REPORT:\n{current_report[:3500]}"
    + f"\n\nCRITIC_FEEDBACK_ITER1:\n{feedback_history[0][:2000]}"
    + f"\n\nCRITIC_FEEDBACK_ITER2:\n{feedback_history[1][:2000]}"
)

final_report = run_single_crewai_step(
    role='QC Reporter',
    goal='Publish final evidence-grounded QC report.',
    backstory='You transform revisions into a clean operational QC plan.',
    prompt=synthesis_prompt,
    expected_output='Final polished report with required sections and exact numbers.',
)

# Final artifact
final_report_path = os.path.join(output_dir, 'qc_report_crewai_llm_final.txt')
with open(final_report_path, 'w', encoding='utf-8') as f:
    f.write('FACT SHEET USED\n')
    f.write('---------------\n')
    f.write(fact_sheet + '\n\n')
    f.write('FINAL REPORT AFTER 2 CRITIC ITERATIONS\n')
    f.write('--------------------------------------\n')
    f.write(final_report)

print('CrewAI two-iteration critic pipeline complete.')
print('Saved files:')
print('-', final_report_path)
print('-', os.path.join(output_dir, 'qc_report_critic_feedback_iter1.txt'))
print('-', os.path.join(output_dir, 'qc_report_revised_iter1.txt'))
print('-', os.path.join(output_dir, 'qc_report_critic_feedback_iter2.txt'))
print('-', os.path.join(output_dir, 'qc_report_revised_iter2.txt'))
print('\nFinal report preview (first 1000 chars):\n')
print(final_report[:1000])

CrewAI two-iteration critic pipeline complete.
Saved files:
- /ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results/qc_report_crewai_llm_final.txt
- /ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results/qc_report_critic_feedback_iter1.txt
- /ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results/qc_report_revised_iter1.txt
- /ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results/qc_report_critic_feedback_iter2.txt
- /ibex/user/sotoorda/masterh1/LLM_project/LLM_model/results/qc_report_revised_iter2.txt

Final report preview (first 1000 chars):

Title: Final Report on Cell Typing Reliability

1. Summary of Reliability:
The FACT_SHEET provides the following numbers related to cell typing: n_cells: 23466, unanimous_3of3: 13012 (55.45%), majority_2of3: 8485 (36.16%), disagreement_1of3: 1969 (8.39%).

2. Exclusion Policy:
The exclusion policy is moderate. We will keep cells with Vote_support in [2,3], but flag Vote_support=2 as medium-confidence.

3. Confidence Tiers for Dow